In [ ]:
import os
from pathlib import Path
from typing import Any

import pandas as pd

from kibad_llm.config import PROJ_ROOT
from kibad_llm.utils.job_return import load_subdirs

# swith to project root to use same paths as in commands
os.chdir(PROJ_ROOT)
# set wider column width for displaying pandas data frames
pd.set_option("max_colwidth", 400)
# show all columns
pd.set_option("display.max_columns", None)

In [ ]:
BASE_DIR = Path("data/prediction_results/159_core_schema_baseline_multirun")
METRICS_DIR = BASE_DIR / "logs/evaluate"
ERRORS_DIR = BASE_DIR / "logs/evaluate_errors"
# don't load this data to declutter the data frames
EXCLUDE_KEYS = [
    "prediction.job_return_value.output_file",
    "overrides.paths.save_dir",
    "overrides.pdf_directory",
]

In [ ]:
metrics_df = pd.DataFrame.from_records(
    load_subdirs(METRICS_DIR, strip_id_keys=True, flatten=True, exclude_keys=EXCLUDE_KEYS)
)
# metrics_df

In [ ]:
# not used


def first_of_same(li: list) -> Any:
    if len(li) == 0:
        raise ValueError("can not get first of empty list")
    if len(set(li)) != 1:
        raise ValueError(f"list contains different entries: {set(li)}")
    return li[0]


def group_by(df: pd.DataFrame, by: str | list[str]):
    if isinstance(by, str):
        by = [by]
    func = {col: list if df[col].dtype == "object" else "mean" for col in df.columns}
    for k in by:
        func[k] = "first"
    result = df.groupby(by, as_index=False).agg(func)
    return result


metrics_df_avg = group_by(metrics_df, by="overrides.extractor/llm")
metrics_df_avg

In [ ]:
cols_to_plot = [
    col for col in metrics_df_avg.columns if col.endswith("f1") and not col.startswith("AVG")
]
metrics_df_avg.plot(kind="bar", x="overrides.extractor/llm", y=cols_to_plot)

In [ ]:
errors_df = pd.DataFrame.from_records(
    load_subdirs(
        ERRORS_DIR,
        strip_id_keys=True,
        flatten=True,
        exclude_keys=[
            "prediction.job_return_value.output_file",
            "overrides.paths.save_dir",
            "overrides.pdf_directory",
        ],
    )
)
# errors_df
errors_df_avg = group_by(errors_df, by="overrides.extractor/llm")
errors_df_avg

In [ ]:
# cols_to_plot = errors_df_avg.select_dtypes(include="number").columns
cols_to_plot = [col for col in errors_df_avg.columns if "error" in str(col) or "Error" in str(col)]
errors_df_avg.plot.bar(stacked=True, x="overrides.extractor/llm", y=cols_to_plot)